# Privacy & Governance

### Data Privacy, Governance, and Compliance Assessment  

The workflow includes the following key components:

1. **Identification of Personally Identifiable Information (PII)**
   - Detects sensitive fields such as names, emails, social security numbers, IP addresses, and dates of birth.

2. **Pseudonymization and Hashing**
   - Protects sensitive data through pseudonymization or hashing, reducing privacy risks while preserving analytical utility.
   - Maintains **audit logs** for all changes to ensure traceability and accountability.

3. **Consent Tracking**
   - Records user consent for data processing, including purpose, method, and timestamp, supporting GDPR compliance.

4. **Data Retention and Storage Limitation**
   - Implements automatic expiration of records via TTL indexes, ensuring that personal data is not stored longer than necessary.

5. **Audit Trails and Oversight**
   - Captures logs of decisions and human oversight for high-risk processes, such as credit scoring.

6. **Regulatory Mapping**
   - Maps dataset fields to relevant **GDPR articles** and references the **EU AI Act** classification for high-risk AI systems, such as credit scoring models.

7. **Automated Compliance Reporting**
   - Generates a detailed report highlighting potential privacy and governance gaps, along with mitigation status and health scores.

8. **Actionable Governance Controls**

The goal of this notebook is to demonstrate **best practices for privacy, data governance, and regulatory compliance** in a structured, auditable, and reproducible way, suitable for both academic analysis and real-world implementation.

## 0. Data Loading

In [ ]:
# Install pymongo
!pip install pymongo
!pip install pandas
!pip install faker  

In [1]:
from pymongo import MongoClient
import json
import os

# Connect to MongoDB
client = MongoClient("mongodb://localhost:27017/")  # Adjust URI if needed
db = client["credit_applications_db"]
collection = db["applications"]


# Load JSON data
file_path = "cleaned_credit_applications.json"  # Must be in cwd
if not os.path.exists(file_path):
    raise FileNotFoundError(f"JSON file not found at {file_path}")

with open(file_path, "r") as f:
    data = json.load(f)

print(f"Loaded {len(data)} records from JSON.")

# Drop collection to start fresh
db.drop_collection("applications")


# Ensure uniqueness of documents in the collection before re-inserting/updating
seen_ids = set()
unique_documents = []

# Fetch all documents currently in the collection
for doc in collection.find():
    if '_id' in doc:
        if doc['_id'] not in seen_ids:
            unique_documents.append(doc)
            seen_ids.add(doc['_id'])
        else:
            print(f"Skipping duplicate document with _id: {doc['_id']} already in the collection.")

# Insert only unique documents
if unique_documents:
    try:
        collection.insert_many(unique_documents)
        print(f"Dataset loaded: {collection.count_documents({})} records ready for audit.\n")
    except Exception as e:
        print(f"Error inserting documents: {e}")
else:
    print("No documents to insert after removing duplicates.\n")

# Insert JSON into MongoDB
collection.insert_many(data)
print(f"Inserted {collection.count_documents({})} documents into MongoDB")

Loaded 500 records from JSON.
No documents to insert after removing duplicates.

Inserted 500 documents into MongoDB


In [5]:
sample = collection.find_one()
print("Sample document keys:", sample.keys())

Sample document keys: dict_keys(['_id', 'spending_behavior', 'processing_timestamp', 'loan_purpose', 'notes', 'applicant_info_full_name', 'applicant_info_email', 'applicant_info_ssn', 'applicant_info_ip_address', 'applicant_info_gender', 'applicant_info_date_of_birth', 'applicant_info_zip_code', 'financials_annual_income', 'financials_credit_history_months', 'financials_debt_to_income', 'financials_savings_balance', 'decision_loan_approved', 'decision_rejection_reason', 'decision_interest_rate', 'decision_approved_amount', 'financials_annual_salary', 'financials_annual_income_canonical', 'analysis_dob_format', 'analysis_dob_parsed', 'analysis_age_years'])
